# Baseline Model - ResNet34 U-Net

Training + evaluation notebook for the baseline segmentation model. This is intentionally a work in progress: use the smoke checks while prototyping, then uncomment full training when the data path and cloud runtime are ready.

## Colab Setup

Run these when using Colab: mounts Drive, moves to repo, install dependencies from `requirements.txt`, and loads W&B auth from Colab Secrets (see README if not setup).

In [1]:
# # Colab only: mount Drive and move into the repo.
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/Deep Learning Project/DLE-Flair-Segmentation

Mounted at /content/drive
/content/drive/MyDrive/Deep Learning Project/DLE-Flair-Segmentation


In [2]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.2 MB/s eta 0:00:00


In [5]:
# Colab only: load W&B API key from Colab Secrets.
# In Colab, add a secret named exactly WANDB_API_KEY before running this cell.
import os
from google.colab import userdata

wandb_api_key = userdata.get("WANDB_API_KEY")
if wandb_api_key:
    os.environ["WANDB_API_KEY"] = wandb_api_key
    print("WANDB_API_KEY loaded from Colab Secrets.")
else:
    print("WANDB_API_KEY was not found in Colab Secrets; W&B training will fail if enabled.")

WANDB_API_KEY loaded from Colab Secrets.


## Local Setup

Start here when local. In Colab, run this section **after** the Colab setup above.

In [3]:
from pathlib import Path
import os
import sys

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "src").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

Repo root: /content/drive/MyDrive/Deep Learning Project/DLE-Flair-Segmentation


In [7]:
import torch
import matplotlib.pyplot as plt

from src.data import build_dataloaders
from src.models import build_model, freeze_backbone
from src.train import (
    _load_env_file,
    build_loss,
    build_optimizer,
    build_scheduler,
    train_from_config,
    train_one_epoch,
    validate_one_epoch,
)
from src.utils import get_device, load_config
#from src.visualize import overlay_mask, prediction_mask

device = get_device()
print(f"Using device: {device}")

Using device: cpu


## Load Config

Uses model specific `yaml` to override defaults for hyperparameters, model weights, architecture setups, etc.

In [8]:
config_path = REPO_ROOT / "configs" / "resnet34_unet.yaml"
config = load_config(config_path)
config["training"]["loss"].setdefault("ignore_index", config["data"].get("ignore_index", 255))

config

{'experiment': {'name': 'resnet34_unet', 'seed': 42, 'output_dir': 'outputs'},
 'data': {'train_csv': 'data/processed/splits/grouped_5class_train.csv',
  'val_csv': 'data/processed/splits/grouped_5class_val.csv',
  'test_csv': 'data/processed/splits/grouped_5class_test.csv',
  'class_map': 'data/processed/class_map.json',
  'train_stats': 'data/processed/train_stats.json',
  'image_size': 512,
  'batch_size': 4,
  'num_workers': 4,
  'channels': [1, 2, 3, 4, 5],
  'norm_type': 'scaling',
  'norm_means': [],
  'norm_stds': [],
  'ignore_index': 255,
  'num_classes': 5,
  'augment': {'enabled': True,
   'horizontal_flip': True,
   'vertical_flip': True,
   'random_rotate_90': True}},
 'model': {'name': 'resnet34_unet',
  'encoder': 'resnet34',
  'pretrained': True,
  'in_channels': 5,
  'num_classes': 5},
 'training': {'max_epochs': 80,
  'min_epochs': 10,
  'warmup_frozen_epochs': 3,
  'early_stopping': {'enabled': True,
   'monitor': 'val_loss',
   'mode': 'min',
   'patience': 8,
   '

## W&B Auth Check

Local runs read `WANDB_API_KEY` from repo-root `.env`. Colab runs should load it from Colab Secrets in the setup cell above. This cell only reports whether a key is available; it never prints the key.

In [ ]:
_load_env_file()
wandb_enabled = config.get("wandb", {}).get("enabled", False)
wandb_key_available = bool(os.environ.get("WANDB_API_KEY"))

print(f"W&B enabled in config: {wandb_enabled}")
print(f"WANDB_API_KEY available: {wandb_key_available}")
if wandb_enabled and not wandb_key_available:
    print("WARN: NOT LOGGED IN TO W&B.\nAdd WANDB_API_KEY to local .env or Colab Secrets before full training.")

## Data Loading

In [9]:
# PLACEHOLDER: dataloader integration point. assumes loader lives in src.data and meets contract below.
# Required contract:
#   train_loader, val_loader, test_loader = build_dataloaders(config["data"])
#   images: [B, 5, H, W] float tensor
#   masks: [B, H, W] long tensor with labels 0..4 and optional ignore_index=255

train_loader, val_loader, test_loader = build_dataloaders(config["data"])
len(train_loader), len(val_loader), len(test_loader)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


(50, 7, 7)

In [10]:
def unpack_batch(batch):
    if isinstance(batch, dict):
        return batch["image"], batch["mask"]
    return batch

batch = next(iter(train_loader))
images, masks = unpack_batch(batch)

print("images", images.shape, images.dtype, images.min().item(), images.max().item())
print("masks ", masks.shape, masks.dtype, torch.unique(masks)[:20])

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


images torch.Size([4, 5, 512, 512]) torch.float32 0.0 1.0
masks  torch.Size([4, 512, 512]) torch.int64 tensor([0, 1, 2, 3, 4])


## Model, Loss, Optimizer

In [11]:
model = build_model(config["model"]).to(device)
loss_fn = build_loss(config["training"]["loss"], device)

warmup_frozen_epochs = config["training"].get("warmup_frozen_epochs", 0)
if warmup_frozen_epochs > 0:
    for p in model.encoder.parameters():
        p.requires_grad = False

optimizer = build_optimizer(model, config["training"]["optimizer"])
scheduler = build_scheduler(
    optimizer,
    config["training"]["scheduler"],
    config["training"]["max_epochs"],
)

print(model.__class__.__name__)
print([group["name"] for group in optimizer.param_groups])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Unet
['head']


In [12]:
# Forward-pass smoke test.
model.eval()
with torch.no_grad():
    sample_logits = model(images[:2].to(device))

print("logits", sample_logits.shape)
assert sample_logits.shape[1] == config["model"]["num_classes"]
assert sample_logits.shape[-2:] == images.shape[-2:]

logits torch.Size([2, 5, 512, 512])


## Single-Batch Verification

Optional. Uncomment when you want to verify one train step and one validation pass. This mutates model weights, so restart the kernel before real training.

In [13]:
train_metrics = train_one_epoch(model, [batch], loss_fn, optimizer, device, epoch=1)
#val_metrics = validate_one_epoch(model, [batch], loss_fn, device)
train_metrics#, val_metrics

{'train_loss': 2.042623996734619}

## Full Training

Optional. Uncomment when the data paths, runtime, and W&B credentials are ready. Checkpoints will be saved under `outputs/resnet34_unet/`.

In [14]:
result = train_from_config(config_path)
result

NotImplementedError: 

## Visualize Predictions

In [15]:
# PLACEHOLDER: run after training or after selecting a checkpoint.
# Missing until a trained checkpoint exists at outputs/resnet34_unet/best.pt.
#
checkpoint_path = REPO_ROOT / "outputs" / "resnet34_unet" / "best.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)
model = build_model(config["model"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
#
with torch.no_grad():
    logits = model(images.to(device))
pred = prediction_mask(logits[0])
#
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(overlay_mask(images[0], masks[0], alpha=0.45))
axes[0].set_title("Ground Truth")
axes[0].axis("off")
axes[1].imshow(overlay_mask(images[0], pred, alpha=0.45))
axes[1].set_title("Prediction")
axes[1].axis("off")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Deep Learning Project/DLE-Flair-Segmentation/outputs/resnet34_unet/best.pt'